# Brain Explorer: a walkthrough

The Brain Explorer is a local viewer for participant-specific neuroimaging: 3-D volume rendering,
linked slices, segmentation boundaries, acquisition-date navigation and explicitly registered
overlays. It reads finished PIE outputs and explicit manifests, and serves them to a browser on
loopback only.

This notebook has two paths through it:

1. **the demo**, on four openly licensed OpenNeuro/niivue participants, which needs no PPMI access;
2. **your own PPMI data**, where the same viewer indexes what `pie.imaging` has already produced.

**No outputs are saved in this notebook, on purpose.** PPMI data is released under a data use
agreement, so a notebook in this repository must not carry participant IDs, image IDs, dates or any
value derived from the download. Every cell is unexecuted; running it on your machine fills the
outputs in there. The viewer shows real IDs on screen, but it reads every one of them from your
local catalogue at runtime, and no ID is written into the source (a test enforces that).

Depth on anything here: [brain_viewer.md](../documentation/brain_viewer.md),
[fmri_viewer.md](../documentation/fmri_viewer.md), and the
[documentation index](../documentation/README.md).

## Before you start

- The imaging environment: `venv_imaging/bin/python -m pip install -r pie/imaging/viewer/requirements.txt`
  (fastapi, uvicorn, nibabel, numpy, scipy, scikit-image, SimpleITK, pandas, httpx).
- The frontend built once: `npm --prefix brain-viewer ci && npm --prefix brain-viewer run build`
  (Node 18+). The API serves `brain-viewer/dist`; if that folder is stale you will see an old UI.
- Run everything **from the repository root**: `pie` is not installed into the environment, so
  `python -m pie.imaging.viewer ...` only resolves from there.
- A browser with WebGL2 (current Chrome/Chromium or Firefox, hardware acceleration on). The API
  itself needs no GPU.
- PPMI access is **not** needed for the demo. For the PPMI path you need the study-data download and
  finished `pie.imaging` outputs.

In [ ]:
from pathlib import Path

REPO = Path.cwd()                      # run this notebook from the repository root
EXAMPLES = REPO / "Imaging/examples"   # demo bundle (gitignored)
DERIVED = REPO / "Imaging/derived"     # PIE's own imaging outputs (gitignored)

for label, path in [
    ("frontend build", REPO / "brain-viewer/dist/index.html"),
    ("viewer requirements", REPO / "pie/imaging/viewer/requirements.txt"),
    ("FastSurfer checkout", REPO / "third_party/FastSurfer/run_fastsurfer.sh"),
    ("demo bundle manifest", EXAMPLES / "manifest.json"),
    ("PIE imaging outputs", DERIVED),
]:
    print(f"{label:<22} {path.exists()}")

## The demo bundle

`scripts/fetch_viewer_examples.py` downloads four openly licensed participants, each file pinned by
URL and SHA-256, and writes `Imaging/examples/manifest.json` plus an `ATTRIBUTION.md` next to the
data. Re-running skips anything that already verifies. About 220 MB.

| Scan | Source | Who it is | What it shows |
|---|---|---|---|
| T1 + 3-D structures | OpenNeuro ds005892 | sub-MJF001, Parkinson's with mild cognitive impairment (PD-MCI), 68, male | FastSurfer-conformed T1, brain mask, DKT + aseg labels |
| Resting BOLD | OpenNeuro ds005892 | the same participant | 200 frames, TR 2 s, native EPI |
| [18F]FE-PE2I PET | OpenNeuro ds006917 | sub-005, healthy control | dopamine-transporter PET, 40-90 min, Bq/mL, averaged to 1 mm |
| DWI | OpenNeuro ds001907 | sub-RC4101, healthy older control | FA over the mean b0, plus MD |
| Head CT | niivue-images (Seg3DData) | clinical context not stated | implanted electrodes, HU |

Only sub-MJF001 has Parkinson's. The PET participant is a healthy control, and the tracer is the
closest open stand-in for DaTscan, **not** DaTscan and not an abnormal scan. Licences and citations:
`Imaging/examples/ATTRIBUTION.md` and
[brain_viewer.md](../documentation/brain_viewer.md#example-data-attribution-and-licences).

In [ ]:
RUN_FETCH = False                 # set True to download ~220 MB into Imaging/examples

import subprocess, sys

FETCH = [sys.executable, "scripts/fetch_viewer_examples.py"]     # --dest DIR, --no-dwi to skip the 67 MB diffusion example

if RUN_FETCH:
    print(subprocess.run(FETCH, cwd=REPO, text=True, capture_output=True).stdout)
else:
    print("template:", " ".join(FETCH))

### The segmentation behind "Inside the brain"

3-D structures need a participant-specific `DKT + aseg` segmentation, so the demo's T1 entry only
appears once FastSurfer has run on it. Segmentation-only FastSurfer needs no FreeSurfer licence;
on CPU it takes roughly 20 minutes. The fetch script prints this command with your absolute paths,
and adds the T1 entry the next time you run it.

The brain mask matters for more than masking noise: it is what keeps the face out of 3-D renders.

In [ ]:
RUN_FASTSURFER = False            # CPU, ~20 minutes; --device cuda if a GPU is free

FASTSURFER = [
    "./run_fastsurfer.sh",
    "--t1", str(EXAMPLES / "ds005892/sub-MJF001_T1w.nii.gz"),
    "--sid", "ds005892_sub-MJF001",
    "--sd", str(EXAMPLES / "fastsurfer"),
    "--seg_only", "--no_cereb", "--no_hypothal", "--no_cc",
    "--device", "cpu", "--viewagg_device", "cpu",
    "--threads", "6", "--py", sys.executable,
]

if RUN_FASTSURFER:
    subprocess.run(FASTSURFER, cwd=REPO / "third_party/FastSurfer", check=True)
    subprocess.run(FETCH, cwd=REPO, check=True)      # adds the T1 + structures entry
else:
    print("template (run from third_party/FastSurfer):", " ".join(FASTSURFER))

In [ ]:
import json

if (EXAMPLES / "manifest.json").exists():
    doc = json.loads((EXAMPLES / "manifest.json").read_text())
    for scan in doc["scans"]:
        print(f"{scan['id']:<20} {scan['modality']:<5} {scan.get('kind', 'scalar'):<10} {Path(scan['path']).name}")
    print("subjects:", [(s["id"], s.get("collection")) for s in doc["subjects"]])
else:
    print("No demo bundle yet - run the fetch cell above.")

## Serving it

```bash
venv_imaging/bin/python -m pie.imaging.viewer serve --manifest Imaging/examples/manifest.json
```

Then open **http://127.0.0.1:8765**. The server binds loopback only, and the cell below is the same
command from Python. Stop it with Ctrl-C in a terminal, or `server.terminate()` here.

| Participant in the sidebar | Try |
|---|---|
| ds005892 sub-MJF001 (PD-MCI) | **Show MRI + structures**, then **Focus on striatum**, then **Review boundaries on MRI slices** |
| the same participant, **fMRI** | click a voxel in a slice for its trace, then **Play**; switch to **Temporal signal-to-noise ratio** |
| ds006917 sub-005 (healthy control) | **Axial**, then widen the colour window; **Four-view** to see all planes |
| ds001907 sub-RC4101 (healthy control) | **Diffusion measure** -> Mean diffusivity; **Four-view** |
| CT_Electrodes | **Bone window**, then rotate the 3-D view |

In [ ]:
RUN_SERVER = False                # set True to start the demo server from this notebook

SERVE_DEMO = [sys.executable, "-m", "pie.imaging.viewer", "serve",
              "--manifest", str(EXAMPLES / "manifest.json")]     # --port N, --cache-dir DIR

if RUN_SERVER:
    server = subprocess.Popen(SERVE_DEMO, cwd=REPO)
    print("serving http://127.0.0.1:8765 - pid", server.pid, "- call server.terminate() when done")
else:
    print("template:", " ".join(SERVE_DEMO))

### What you should see

![Brain-masked T1 with the participant's caudate and putamen boundaries](../assets/screenshots/brain_viewer_structures.png)

*sub-MJF001 (PD-MCI): his own T1 at 35% opacity with his FastSurfer caudate and putamen boundaries.
Estimated segmentation boundaries, not validated surfaces. The mask removes the face.*

![Four-view with striatal outlines on the measured slices](../assets/screenshots/brain_viewer_fourview.png)

*The same selection in Four-view: linked axial, coronal and sagittal slices with the selected labels
outlined on measured MRI, and the 3-D panel. Meshes are never drawn into the slice views.*

![Resting BOLD frame with playback transport and a fixed-voxel trace](../assets/screenshots/brain_viewer_fmri.png)

*The same participant's resting BOLD: one frame of 200, the transport, and every sample at one fixed
voxel. Unprocessed native EPI - no motion, slice-timing or distortion correction, and not an
activation map. Within-scan seconds are not visit dates.*

![Axial PET slice with striatal dopamine-transporter uptake](../assets/screenshots/brain_viewer_pet.png)

*A **healthy control** (ds006917 sub-005): [18F]FE-PE2I uptake concentrates in the caudate and
putamen. This is PET, not DaTscan SPECT, and not a Parkinson's scan.*

![Fractional anisotropy over the mean b0](../assets/screenshots/brain_viewer_dti.png)

*A healthy older control (ds001907 sub-RC4101): FA over the motion-corrected mean b0, computed with
`pie.imaging.dwi`. FA measures diffusion anisotropy, not tract count.*

![Head CT in the bone window showing implanted electrodes](../assets/screenshots/brain_viewer_ct.png)

*Head CT with implanted electrodes in the bone window (-400 to 1800 HU). An MIT-licensed example
image; its clinical context is not stated.*

## The same viewer on your PPMI data

Drop `--manifest` and the viewer indexes what `pie.imaging` has already written. Nothing is
recomputed and nothing is modified: discovery is read-only.

| It reads | For |
|---|---|
| `Imaging/derived/sessions.csv` + `Imaging/derived/fastsurfer/<IMAGE_ID>/mri/{orig_nu.mgz or orig.mgz, mask.mgz, aparc.DKTatlas+aseg.deep.mgz}` | one MRI per finished FastSurfer subject, with mask and DKT + aseg labels |
| `Imaging/derived/dwi/dwi_index.csv` + `Imaging/derived/dwi/<PATNO>/{fa,b0,md,fw,fat,aseg_dwi}.nii.gz` | FA over the mean b0, with MD / FW / FAt as extra measures |
| `Imaging/derived/datscan_v5/datscan_sbr.csv` + `Imaging/First_Study_SPECT_<date>.csv` | reconstructed DaTscan volumes, with their acquisition dates and visits |
| `Imaging/derived/datscan_full/datscan_sbr.csv` | the explicitly **unreviewed** SPECT-in-MRI alignment preview |
| `<ppmi-dir>/_Subject_Characteristics/Participant_Status_<date>.csv` | the current cohort of each participant |
| `Imaging/derived/viewer_collection/manifest.json` | extra scans, loaded automatically when present |

Scan IDs are derived, never typed: `mri-<IMAGE_ID>`, `dti-<PATNO>`, `spect-<IMAGE_ID>`. PPMI
re-releases its tables under new dated names, so the viewer takes the **newest** release matching
each stem. A participant enters the index through a finished MRI; diffusion and SPECT rows without
one are skipped and counted in a warning. Anything missing is a warning in `/api/catalog` and the
sidebar, never an exception.

In [ ]:
RUN_SERVER_PPMI = False           # your own data; add --ppmi-dir if the study download is elsewhere

SERVE_PPMI = [sys.executable, "-m", "pie.imaging.viewer", "serve"]   # --ppmi-dir DIR, --cache-dir DIR, --port N

if RUN_SERVER_PPMI:
    server = subprocess.Popen(SERVE_PPMI, cwd=REPO)
    print("serving http://127.0.0.1:8765 - pid", server.pid)
else:
    print("template:", " ".join(SERVE_PPMI))

### Which scans to request next

`sample-plan` reads the local acquisition tables (`MRI_Acquisition_Metadata_<date>.csv`,
`DaTScan_Acquisition_Metadata_<date>.csv`, `DaTscan_Imaging_<date>.csv`,
`PET_Acquisition_Metadata_<date>.csv`, `CT_Scan_<date>.csv`, `Safety_Head_CT_Scan_<date>.csv`) plus
your local index, and writes a shortlist you can paste into IDA:

```bash
venv_imaging/bin/python -m pie.imaging.viewer sample-plan --ppmi-dir /path/to/PPMI/study_data
```

| Output | What it holds |
|---|---|
| `plan.json` | `coverage`, `shortlist`, `next_bundle`, `archive_groups`, `warnings`, `notes` |
| `download_candidates.csv` | the evidence row behind each suggested PATNO |
| `<MODALITY>_subject_ids.txt` | comma-separated PATNOs for IDA's Subject ID field |
| `DOWNLOAD_PLAN.md` | the same plan to read, including its warnings |

`serve` reads `plan.json` from the same `--output` folder and hands it to the browser at
`/api/sample-plan`; the **Sample plan** tab renders it. `next_bundle` is the panel at the top:
up to three shortlisted participants whose candidate modalities are not local yet, most incomplete
first, each with the dates found in the tables. Those PATNOs come from your tables at runtime.
`--if-missing` keeps an existing plan instead of rebuilding it, which is what
`scripts/run_brain_viewer.sh` uses.

A clinical form is evidence that an acquisition happened, not that an image is downloadable, so
every row says "confirm in IDA". Month-precision dates are search hints, not acquisition timestamps.

In [ ]:
RUN_SAMPLE_PLAN = False

SAMPLE_PLAN = [sys.executable, "-m", "pie.imaging.viewer", "sample-plan"]   # --ppmi-dir DIR, --output DIR, --if-missing

if RUN_SAMPLE_PLAN:
    print(subprocess.run(SAMPLE_PLAN, cwd=REPO, text=True, capture_output=True).stdout)
else:
    print("template:", " ".join(SAMPLE_PLAN))

# Counts only: a plan built from PPMI tables lists real PATNOs, which do not belong in a notebook.
PLAN = DERIVED / "viewer_sample_plan/plan.json"
if PLAN.exists():
    plan = json.loads(PLAN.read_text())
    print("sections:", sorted(plan))
    print("coverage rows:", len(plan.get("coverage", [])), "| shortlist rows:", len(plan.get("shortlist", [])))
    print("next_bundle entries:", len(plan.get("next_bundle", [])), "| warnings:", len(plan.get("warnings", [])))
else:
    print("no plan at", PLAN)

### Bringing in scans PIE has not processed

Two helpers convert explicitly named series from a local IDA archive and merge them into
`Imaging/derived/viewer_collection/manifest.json`, which `serve` loads automatically. Both merge by
scan ID and record a source fingerprint, so they can share one collection folder in either order,
and re-running with the same sources is a no-op for the conversion.

| Script | Flags | Gives you |
|---|---|---|
| `scripts/prepare_viewer_followup.py` | `--archive`, `--collection`, `--subject`, `--images <A> <B>`, `--output` | two dated T1 visits of one participant, as native head MRIs |
| `scripts/prepare_viewer_fmri.py` | `--archives`, `--collection`, `--images`, `--output` | explicitly chosen BOLD runs, with their converter receipts |

**Compare visits** needs exactly that: two MRIs of one participant on two distinct dates. Different
modalities on different dates are not a follow-up. When the selected participant has only one date,
the panel names a participant your index does have two dated MRIs for - again, read from the
catalogue, not from source. The optional rigid alignment it offers is marked **unreviewed**: it
resamples the follow-up into baseline geometry for visual inspection and establishes nothing
quantitative.

In [ ]:
RUN_PREPARE = False               # fill in your own archive, collection CSV, PATNO and image IDs

PREPARE_FOLLOWUP = [
    sys.executable, "scripts/prepare_viewer_followup.py",
    "--archive", "/path/to/collection.zip",
    "--collection", "/path/to/collection.csv",
    "--subject", "<PATNO>",
    "--images", "<IMAGE_ID_A>", "<IMAGE_ID_B>",
    "--output", str(DERIVED / "viewer_collection"),
]

if RUN_PREPARE:
    subprocess.run(PREPARE_FOLLOWUP, cwd=REPO, check=True)
else:
    print("template:", " ".join(PREPARE_FOLLOWUP))

## The import manifest

Anything the viewer does not discover itself comes from a version-1 manifest: other modalities,
segmentations, tractography, non-PPMI data. Paths resolve against the manifest's own folder.

| Field | Meaning |
|---|---|
| `id`, `subject`, `modality`, `path` | required; modality is MRI, DTI, SPECT, PET, CT or fMRI |
| `date` | a real `YYYY-MM-DD`, or null. Never invented |
| `kind` | `scalar`, `timeseries` (4D) or `tracts` |
| `space` | the coordinate space you declare; fusion compares it literally |
| `mask`, `atlas`, `atlas_name`, `atlas_lut` | brain mask and integer labels on the same grid; `DKT + aseg` uses the bundled lookup table |
| `anatomy`, `extra` | an underlay, and extra maps on the primary grid (the DTI measure selector) |
| `registration`, `reference_id` | `native` or `verified`; `verified` needs a same-subject reference in the same space |
| `units`, `tracer`, `qc`, `provenance`, `metadata` | shown in the UI and written into exports |

Validation happens before the server starts, and names the file and scan: unknown fields, missing
files, non-ISO dates, duplicate IDs and bad `verified` references are all refused there.

In [ ]:
import tempfile

import nibabel as nib
import numpy as np

from pie.imaging.viewer.catalog import Catalog

tmp = Path(tempfile.mkdtemp(prefix="viewer-walkthrough-"))

def volume(name, shape=(8, 8, 8)):
    image = nib.Nifti1Image(np.random.default_rng(0).uniform(0, 100, shape).astype("float32"), np.eye(4))
    image.header.set_xyzt_units("mm")
    nib.save(image, tmp / name)
    return name

doc = {
    "version": 1,
    "subjects": [{"id": "P001", "collection": "Synthetic", "group": "Example", "cohort": "Example"}],
    "scans": [
        {"id": "p001-t1", "subject": "P001", "modality": "MRI", "date": "2000-01-01",
         "path": volume("t1.nii.gz"), "space": "sub-P001:T1", "units": "arbitrary intensity"},
        {"id": "p001-pet", "subject": "P001", "modality": "PET", "date": "2000-01-02",
         "path": volume("pet.nii.gz"), "space": "sub-P001:T1", "tracer": "synthetic",
         "units": "synthetic units", "registration": "verified", "reference_id": "p001-t1"},
    ],
}
(tmp / "manifest.json").write_text(json.dumps(doc))

catalog = Catalog(tmp / "repo", tmp / "no-ppmi", tmp / "manifest.json")
print("scans:", sorted(catalog.scans), "| subjects:", sorted(catalog.subjects))
print("PET registration:", catalog.scans["p001-pet"].registration,
      "-> reference:", catalog.scans["p001-pet"].reference_id)
print("index warnings:", len(catalog.warnings))

### Why fusion is refused by default

A valid affine says how a volume sits in its own scanner's coordinates. It does not say that two
images of one person line up. So `registration: "native"` - the default - means the scan is shown
on its own, and the **Registered overlay** menu only offers images that are the same participant, in
the same declared `space`, marked `verified`, and pointing at the active scan with `reference_id`.

`verified` is your declaration that you registered the images outside the viewer, saved the
transformed image and checked it. The viewer never estimates a transform to make an overlay look
plausible. Its two automatic alignments - the SPECT-in-MRI preview and the rigid Compare visits
preview - are always labelled unreviewed and never feed that menu.

Break one of those conditions and the manifest is refused at load, with the reason:

In [ ]:
broken = json.loads((tmp / "manifest.json").read_text())
broken["scans"][1]["space"] = "sub-P001:PET-native"      # verified, but no longer the reference's space
(tmp / "broken.json").write_text(json.dumps(broken))

try:
    Catalog(tmp / "repo", tmp / "no-ppmi", tmp / "broken.json")
except ValueError as error:
    print("refused:", error)

## The service itself

- **Loopback only.** It binds `127.0.0.1`, and a host check accepts only `127.0.0.1`, `localhost`
  and `[::1]`, so a page on the internet cannot reach it by DNS rebinding. There is no
  authentication layer, and no telemetry or external request of any kind.
- **Files by opaque ID.** Volumes are served as `/api/assets/<sha256 of the path>/<filename>`, and
  only after the scan that owns them has been prepared. There is no path parameter to point
  anywhere else.
- **A cache, not a rewrite.** Prepared copies (MGZ converted to NIfTI, masked volumes, label
  atlases, meshes, BOLD summaries, comparison resamples) live under `Imaging/derived/viewer_cache/`,
  or wherever `--cache-dir` points, keyed by the scan record plus each source's size and mtime.
  Sources are never modified. `--require-cache-mount` refuses to start unless that directory is a
  real mount point, and returns 503 rather than silently filling the parent disk if it disappears.

In [ ]:
from fastapi.testclient import TestClient

from pie.imaging.viewer.server import create_app

with tempfile.TemporaryDirectory() as empty:
    app = create_app(Path(empty), Path(empty) / "no-ppmi", cache=Path(empty) / "cache")
    for path in sorted(app.openapi()["paths"]):
        print(path)

    client = TestClient(app)
    print()
    print("health:", client.get("/api/health").json())
    print("catalog keys:", sorted(client.get("/api/catalog").json()))
    print("index warnings with no data:", len(client.get("/api/catalog").json()["warnings"]))
    print("api cache header:", client.get("/api/catalog").headers["cache-control"])
    print("foreign Host header ->", client.get("/api/health", headers={"host": "evil.example"}).status_code)

Against a running server, the same endpoints answer over HTTP. `/api/scans/{id}` is the one that
does work: it prepares the display copy and returns the volume list, the region table, the geometry
and a `fingerprint` that identifies that exact acquisition and processing.

In [ ]:
RUN_API_QUERY = False             # start the demo server first (RUN_SERVER above)

DEMO_URL = "http://127.0.0.1:8765"

if RUN_API_QUERY:
    import httpx

    print("health:", httpx.get(f"{DEMO_URL}/api/health").json())
    catalog = httpx.get(f"{DEMO_URL}/api/catalog").json()
    print("subjects:", len(catalog["subjects"]), "| scans:", catalog["scan_count"],
          "| warnings:", len(catalog["warnings"]))

    # Counts and roles only: a PPMI catalogue lists real IDs.
    scan_id = catalog["subjects"][0]["scans"][0]["id"]
    prepared = httpx.get(f"{DEMO_URL}/api/scans/{scan_id}", timeout=300).json()
    print("volume roles:", [v["role"] for v in prepared["volumes"]],
          "| regions:", len(prepared["regions"]),
          "| frames:", prepared["geometry"]["frames"])
else:
    print("template - start the server, then set RUN_API_QUERY = True")

## Recording what you saw

Two things leave the viewer, and neither is a patient image in a shared file:

- **Review notes.** Reviewer, planes inspected, a visual observation and free text, stored in that
  browser under `pie-review-v1:<fingerprint>`, so a note stays bound to the exact acquisition and
  processing it was written against. Export the history as JSON. A note never changes a scan's
  registration status or enables fusion.
- **Exports.** The camera button writes a PNG whose header carries the participant, dates,
  unreviewed status, representation, window and units, and the fingerprint. **View state** writes
  JSON with the display settings and provenance but no image data. Compare visits exports its
  registration record, and the fMRI workbench exports a voxel trace (with the sampled voxel centre
  in RAS) and the descriptive diagnostics.

Both carry whatever the scan is marked with, which is why the SPECT preview and the rigid comparison
say **ALIGNMENT NOT REVIEWED** on screen and in the export.

## When something looks wrong

| Symptom | Cause and fix |
|---|---|
| The UI looks out of date, or a new control is missing | `brain-viewer/dist` is a stale build. `npm --prefix brain-viewer run build` |
| The **Next bundle** panel says there is no plan | No `plan.json` in the `--output` folder, or one built before `next_bundle` existed. Re-run `sample-plan` |
| The sidebar shows index warnings | Expected when a table or output folder is missing: the viewer reports it and indexes what it can, instead of failing |
| Every `/api/*` call returns 503 | `--require-cache-mount` is set and the cache filesystem is not mounted. Remount it; nothing falls back to the parent disk |
| The 3-D view goes black, or lighting cannot be switched on | The driver lost the WebGL context during the gradient-lighting pass. Lighting is off by default, and the viewer rebuilds the renderer once without it |
| The dev server's `/api` calls 404 | `npm run dev` proxies to port 8765; if the API runs elsewhere, set `PIE_VIEWER_API_PORT` to match |
| A manifest is refused at startup | Read the message: it names the file, the scan and the field. Nothing is served from a half-valid manifest |

## Where to go next

- **Full reference.** [brain_viewer.md](../documentation/brain_viewer.md) documents every flag,
  endpoint, manifest field, cache folder and fusion rule, plus the open-data quickstart this
  notebook follows. [fmri_viewer.md](../documentation/fmri_viewer.md) covers the BOLD workbench:
  what the temporal mean, SD, tSNR and raw DVARS are computed from, and what they are not.
- **The imaging layer that produces what the viewer reads.**
  [imaging.md](../documentation/imaging.md), and the
  [documentation index](../documentation/README.md) for the rest.
- **Tests, as executable documentation.** `tests/test_brain_viewer.py`,
  `tests/test_viewer_anatomy.py`, `tests/test_viewer_structures.py`,
  `tests/test_viewer_comparison.py`, `tests/test_viewer_fmri.py`, `tests/test_viewer_cache.py` and
  `tests/test_viewer_scripts.py` run in seconds on synthetic arrays:
  `venv_imaging/bin/python -m pytest tests/test_brain_viewer.py tests/test_viewer_*.py -q`.

Before sharing this notebook: **Kernel -> Restart & Clear Output**. The data use agreement covers
anything computed from the download, cell outputs included.